<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/05_Recommendation_Systems/01_Persona_Aware_Recommender_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5: Persona-Aware Hybrid Basket Recommender

## Research Objective
To investigate whether customer-segment-specific association rules can improve next-item basket recommendations compared with non-personalized popularity-based baselines.

**H0:** Segment-aware recommendation does not improve ranking performance over baseline methods.
**H1:** Segment-aware recommendation improves ranking performance over baseline methods.

## Step 1: Solving Information Leakage (The Temporal Split)
If a recommendation engine is tested on the exact same data used to discover its underlying association rules, the evaluation is academically invalid (Information Leakage).

To simulate a real-world deployment, we will construct a strict temporal split:
*   **Training Data (First 80% of chronological transactions):** Used to learn shopping behavior, calculate popularity baselines, and mine persona-specific association rules.
*   **Testing Data (Final 20% of chronological transactions):** Used exclusively to build "hidden-item" baskets to evaluate our recommendation algorithms using `Precision@K`, `Recall@K`, and `NDCG@K`.

In [1]:
# Install the Dunnhumby Complete Journey dataset package
!pip install completejourney_py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 47.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import ast
from completejourney_py import get_data

# 1. Fetch the raw data using your library
data = get_data()
transactions = data["transactions"]
products = data["products"]

# 2. Loading the Master Cluster labels from Phase 3
clusters = pd.read_csv('master_customers_fully_clustered.csv')

# 3. Map the cluster labels and product categories onto the raw transactions
basket_data = transactions.merge(clusters[['household_id', 'Hierarchical_Cluster']], on='household_id', how='inner')
basket_data = basket_data.merge(products[['product_id', 'product_category']], on='product_id', how='left')

# 4. Sort the entire dataset chronologically to simulate time
basket_data['transaction_timestamp'] = pd.to_datetime(basket_data['transaction_timestamp'])
basket_data = basket_data.sort_values('transaction_timestamp')

# 5. Execute the 80/20 Temporal Split
split_idx = int(len(basket_data) * 0.8)
train_data = basket_data.iloc[:split_idx].copy()
test_data = basket_data.iloc[split_idx:].copy()

print("✅ Temporal Split Complete!")
print(f"Training Data (Learning): {len(train_data):,} individual item scans")
print(f"Testing Data (Evaluation): {len(test_data):,} individual item scans")

✅ Temporal Split Complete!
Training Data (Learning): 663,080 individual item scans
Testing Data (Evaluation): 165,770 individual item scans


## Step 2: Constructing the Fallback Hierarchy (Baselines)

To handle the "Cold Start" problem and to establish benchmarks for our offline evaluation, we calculate simple popularity metrics derived exclusively from the training data.

The recommendation engine will utilize a fallback hierarchy:
1. **Targeted:** Persona-Specific Association Rules (If cart data and persona are known).
2. **Fallback 1:** Persona Popularity (If the cart is empty or yields no rules, but the user's demographic is known).
3. **Fallback 2:** Global Popularity (If the user is entirely unknown).

In [3]:
# 1. Baseline 1: Global Popularity
# (The top 10 most frequently purchased categories across all users)
global_popularity = train_data['product_category'].value_counts().head(10).index.tolist()

# 2. Baseline 2: Persona Popularity
# (The top 10 most frequently purchased categories per cluster)
persona_popularity = {}
for cluster_id in sorted(train_data['Hierarchical_Cluster'].dropna().unique()):
    # Isolate the training data for this specific cluster
    cluster_subset = train_data[train_data['Hierarchical_Cluster'] == cluster_id]

    # Calculate their specific top 10 items
    top_items = cluster_subset['product_category'].value_counts().head(10).index.tolist()
    persona_popularity[cluster_id] = top_items

print("🎯 Fallback Baselines Established!\n")
print(f"Global Top 3: {global_popularity[:3]}")

# Display the top 3 items for each persona to ensure they captured distinct behaviors
for cluster_id, items in persona_popularity.items():
    print(f"Cluster {int(cluster_id)} Top 3: {items[:3]}")

🎯 Fallback Baselines Established!

Global Top 3: ['SOFT DRINKS', 'FLUID MILK PRODUCTS', 'BAKED BREAD/BUNS/ROLLS']
Cluster 0 Top 3: ['SOFT DRINKS', 'FLUID MILK PRODUCTS', 'BAKED BREAD/BUNS/ROLLS']
Cluster 1 Top 3: ['SOFT DRINKS', 'BAKED BREAD/BUNS/ROLLS', 'FLUID MILK PRODUCTS']
Cluster 2 Top 3: ['SOFT DRINKS', 'FLUID MILK PRODUCTS', 'BAKED BREAD/BUNS/ROLLS']


## Step 3: Mining Persona-Specific Rules (Training)

With our training data isolated, we now rebuild our one-hot encoded matrices and run the Apriori algorithm for **all three clusters** (0, 1, and 2).

Unlike the naive popularity baselines, these rules rely on **Lift** and **Confidence** to identify products that are fundamentally tied together in the minds of specific shopper demographics. These rules will form the primary logic layer of our Hybrid Recommender.

In [4]:
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) # Hides those Google Colab timezone warnings!

# 1. Helper function for One-Hot Encoding
def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1

# 2. Initialize a dictionary to store the trained rules for each persona
persona_rules = {}

print("⚙️ Mining uncontaminated association rules from Training Data...\n")

# 3. Loop through each cluster and generate their specific rules
for cluster_id in sorted(train_data['Hierarchical_Cluster'].dropna().unique()):
    print(f"Processing Cluster {int(cluster_id)}...")

    # Isolate cluster data
    cluster_subset = train_data[train_data['Hierarchical_Cluster'] == cluster_id]

    # Pivot into baskets
    basket = (cluster_subset
              .groupby(['basket_id', 'product_category'])['quantity']
              .sum().unstack().reset_index().fillna(0)
              .set_index('basket_id'))

    # Apply boolean encoding
    basket_sets = basket.apply(lambda x: x.map(encode_units))

    # Run Apriori (3% support)
    frequent_itemsets = apriori(basket_sets, min_support=0.03, use_colnames=True)

    # Generate Association Rules (Minimum Lift > 1.0 to ensure positive correlation)
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

    # Store the rules dataframe in our dictionary
    persona_rules[cluster_id] = rules
    print(f" -> Discovered {len(rules):,} rules for Cluster {int(cluster_id)}.")

print("\n✅ Persona-Specific Rule Mining Complete!")

⚙️ Mining uncontaminated association rules from Training Data...

Processing Cluster 0...
 -> Discovered 410 rules for Cluster 0.
Processing Cluster 1...
 -> Discovered 322 rules for Cluster 1.
Processing Cluster 2...
 -> Discovered 1,664 rules for Cluster 2.

✅ Persona-Specific Rule Mining Complete!


## Explanation
The volume of rules across the clusters perfectly mirrors what we found in Phase 4: Cluster 1 has a narrow, highly focused shopping list (322 rules), while Cluster 2 explores the entire store for complex meal solutions (1,664 rules).

Now, we get to engineer the brain of the operation.

We are going to build the Persona-Aware Hybrid Basket Recommender class. This engine will take a simulated digital cart and a cluster label, and output exactly 3 recommendations.

## Step 4: Building the Hybrid Recommendation Engine

We now construct the `HybridBasketRecommender` class. This engine executes a logical hierarchy to generate Top-K recommendations:

1. **Rule Matching:** It scans the user's cart against the persona-specific association rules.
2. **Scoring:** It calculates a composite score for each triggered rule ($Score = Confidence \times Lift$) and aggregates the scores for overlapping consequents.
3. **Filtering:** It explicitly prevents recommending items already present in the cart.
4. **Fallback (Graceful Degradation):** If the association rules yield fewer than K recommendations (or if the user is unknown), the engine backfills the remaining slots using Persona Popularity, followed by Global Popularity.

In [5]:
class HybridBasketRecommender:
    def __init__(self, global_pop, persona_pop, rules_dict):
        self.global_pop = global_pop
        self.persona_pop = persona_pop
        self.rules_dict = rules_dict

    def recommend(self, cart, persona=None, k=3):
        cart_set = set(cart)
        recommendations = {}

        # STEP 1: If we know the persona, try Association Rules first
        if persona is not None and persona in self.rules_dict:
            rules = self.rules_dict[persona]

            # Find rules where the antecedent is a subset of the current cart
            matching_rules = rules[rules['antecedents'].apply(lambda ant: ant.issubset(cart_set))]

            # Aggregate evidence: Score = Confidence * Lift
            for _, row in matching_rules.iterrows():
                rule_score = row['confidence'] * row['lift']

                # A rule might recommend multiple items; we extract each one
                for item in row['consequents']:
                    if item not in cart_set: # Prevent recommending items already in cart
                        recommendations[item] = recommendations.get(item, 0) + rule_score

        # Sort the rule-based recommendations by their aggregated score
        sorted_recs = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)
        final_recs = [item for item, score in sorted_recs]

        # STEP 2: Fallback to Persona Popularity if we don't have enough rules
        if len(final_recs) < k and persona is not None and persona in self.persona_pop:
            for item in self.persona_pop[persona]:
                if item not in cart_set and item not in final_recs:
                    final_recs.append(item)
                if len(final_recs) == k:
                    break

        # STEP 3: Fallback to Global Popularity (Cold Start / Unknown User)
        if len(final_recs) < k:
            for item in self.global_pop:
                if item not in cart_set and item not in final_recs:
                    final_recs.append(item)
                if len(final_recs) == k:
                    break

        return final_recs[:k]

# Initialize the engine with our trained baselines and rules
recommender = HybridBasketRecommender(
    global_pop=global_popularity,
    persona_pop=persona_popularity,
    rules_dict=persona_rules
)
print("✅ HybridBasketRecommender Engine Initialized and Ready!")

✅ HybridBasketRecommender Engine Initialized and Ready!
